In [ ]:
import pandas as pd
import numpy as np
# load data
df = pd.read_csv('../data/cleaned/wednesday_cleaned.csv')
print(f'Shape: {df.shape}')

# get columns that are numberic 
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Numeric columns: {len(numeric_cols)}')

Shape: (61001, 69)
Numeric columns: 68


In [5]:
# compute mean, std, min, max for all features
stats = pd.DataFrame({
    'mean': df[numeric_cols].mean(),
    'std': df[numeric_cols].std(),
    'min': df[numeric_cols].min(),
    'max': df[numeric_cols].max()
})

print('Stats:')
print(stats)

Stats:
                                     mean           std  min          max
Destination_Port             7.592881e+02  5.734532e+03  0.0      63276.0
Flow_Duration                4.189280e+07  4.263155e+07  1.0  119991902.0
Total_Fwd_Packets            5.866887e+00  7.099562e+01  1.0      17487.0
Total_Backward_Packets       4.246668e+00  9.556690e+01  0.0      23539.0
Total_Length_of_Fwd_Packets  3.848013e+02  1.811250e+03  0.0     278248.0
...                                   ...           ...  ...          ...
Idle_Mean                    3.677495e+07  4.274586e+07  0.0  120000000.0
Idle_Std                     9.960781e+05  5.291552e+06  0.0   60800000.0
Idle_Max                     3.775972e+07  4.280229e+07  0.0  120000000.0
Idle_Min                     3.596751e+07  4.304179e+07  0.0  120000000.0
Attack                       9.180833e-01  2.742400e-01  0.0          1.0

[68 rows x 4 columns]


In [3]:
# identify low and zero variance features
variance = df[numeric_cols].var()

zero_var = []
low_var = []

for col in numeric_cols:
    var_val = variance[col]
    if var_val == 0:
        zero_var.append(col)
    elif var_val < 0.01:
        low_var.append(col)

print(f'Zero variance features: {len(zero_var)}')
print(zero_var)

print(f'\nLow variance features (var < 0.01): {len(low_var)}')
for col in low_var:
    print(f'  {col}: variance = {variance[col]:.6f}')

Zero variance features: 0
[]

Low variance features (var < 0.01): 2
  RST_Flag_Count: variance = 0.000033
  ECE_Flag_Count: variance = 0.000033


In [6]:
# check which features are skewed
skewness = df[numeric_cols].skew()

# find features with high skewness absolute val over 1
skewed_features = []
skew_values = []

for col in numeric_cols:
    skew_val = skewness[col]
    if skew_val > 1 or skew_val < -1:
        skewed_features.append(col)
        skew_values.append(skew_val)

# sort by absolute value of skewness with highest first
sorted_pairs = sorted(zip(skewed_features, skew_values), key=lambda x: abs(x[1]), reverse=True)
skewed_features = [pair[0] for pair in sorted_pairs]
skew_values = [pair[1] for pair in sorted_pairs]

print('Highly skewed features (skew > 1 or < -1):', len(skewed_features))
print('\nTop 10 most skewed:')
for i in range(10):
    if i < len(skewed_features):
        print(str(i+1) + '. ' + skewed_features[i] + ': ' + str(round(skew_values[i], 3)))

Highly skewed features (skew > 1 or < -1): 48

Top 10 most skewed:
1. act_data_pkt_fwd: 245.906
2. Subflow_Bwd_Bytes: 245.899
3. Total_Length_of_Bwd_Packets: 245.899
4. Total_Backward_Packets: 244.864
5. Subflow_Bwd_Packets: 244.864
6. Total_Fwd_Packets: 244.75
7. Subflow_Fwd_Packets: 244.75
8. Bwd_Header_Length: 243.212
9. Fwd_Header_Length: 242.962
10. RST_Flag_Count: 174.64


12/1/2025 - Umar - Univariate Analysis

**Purpose:**  
Compute summary statistics for all numeric features and identify possible issues with variance and distribution skewness.

**Interpretation / Findings:** 
- **Basic Statistics:** Computed mean, std, min, and max for all features
- **Zero Variance Features:** 0 features with zero variance (all features have variation)
- **Low Variance Features:** 2 features with variance < 0.01:
  - RST_Flag_Count: variance = 0.000033
  - ECE_Flag_Count: variance = 0.000033
  - Not sure if these would be useful for modeling in this context
- **Skewed Distributions:** Many features show high skewness (|skew| > 1)
  - Features related to flow bytes, packet sizes, and timing show extreme skewness
  - I feel like this may be expected in network traffic 
  - May need transformation 

**Notes for Team:**
Consider removing or combining the low variance flagged features. Skewed features may need some kind of transformation before modeling.

### Done with Task

In [7]:
import pandas as pd
 
# Load the cleaned dataset
df = pd.read_csv("../data/cleaned/wednesday_cleaned.csv")
 
# Display the shape
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
 
# Display all feature names
print("\nFeature Names:\n")
for col in df.columns:
    print(col)
 
# Display first 5 rows
df.head()

Number of rows: 61001
Number of columns: 69

Feature Names:

Destination_Port
Flow_Duration
Total_Fwd_Packets
Total_Backward_Packets
Total_Length_of_Fwd_Packets
Total_Length_of_Bwd_Packets
Fwd_Packet_Length_Max
Fwd_Packet_Length_Min
Fwd_Packet_Length_Mean
Fwd_Packet_Length_Std
Bwd_Packet_Length_Max
Bwd_Packet_Length_Min
Bwd_Packet_Length_Mean
Bwd_Packet_Length_Std
Flow_Bytes/s
Flow_Packets/s
Flow_IAT_Mean
Flow_IAT_Std
Flow_IAT_Max
Flow_IAT_Min
Fwd_IAT_Total
Fwd_IAT_Mean
Fwd_IAT_Std
Fwd_IAT_Max
Fwd_IAT_Min
Bwd_IAT_Total
Bwd_IAT_Mean
Bwd_IAT_Std
Bwd_IAT_Max
Bwd_IAT_Min
Fwd_PSH_Flags
Fwd_Header_Length
Bwd_Header_Length
Fwd_Packets/s
Bwd_Packets/s
Min_Packet_Length
Max_Packet_Length
Packet_Length_Mean
Packet_Length_Std
Packet_Length_Variance
FIN_Flag_Count
SYN_Flag_Count
RST_Flag_Count
PSH_Flag_Count
ACK_Flag_Count
URG_Flag_Count
ECE_Flag_Count
Down/Up_Ratio
Average_Packet_Size
Avg_Fwd_Segment_Size
Avg_Bwd_Segment_Size
Subflow_Fwd_Packets
Subflow_Fwd_Bytes
Subflow_Bwd_Packets
Subflow_Bwd_B

,Destination_Port,Flow_Duration,Total_Fwd_Packets,Total_Backward_Packets,Total_Length_of_Fwd_Packets,Total_Length_of_Bwd_Packets,Fwd_Packet_Length_Max,Fwd_Packet_Length_Min,Fwd_Packet_Length_Mean,Fwd_Packet_Length_Std,...,Active_Mean,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label,Attack
0,443,87261,1,1,0,0,0,0,0.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
1,443,523,2,0,0,0,0,0,0.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
2,53,94304,1,1,56,338,56,56,56.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
3,53,207,2,2,84,246,42,42,42.0,0.0,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
4,123,69022032,2,2,96,96,48,48,48.0,0.0,...,22276.0,0.0,22276,22276,69000000.0,0.0,69000000,69000000,BENIGN,0


For feature selection, we started with all numeric columns in the dataset. Then we removed a few groups of columns that are not useful for univariate analysis.

First, we dropped the columns “Label” and “Attack.” Label is a text field and Attack is the target we want to predict, so we do not include them in EDA.

Next, we dropped the TCP flag columns because they only contain 0s and 1s. These do not create real distributions for histograms or boxplots, so they are not helpful. These columns include FIN_Flag_Count, SYN_Flag_Count, RST_Flag_Count, PSH_Flag_Count, ACK_Flag_Count, URG_Flag_Count, ECE_Flag_Count, and Fwd_PSH_Flags.

We also removed any columns that were constant. A column that never changes cannot show any pattern.

Finally, we removed columns that had negative values. Negative values are not possible for these network features, so we dropped the columns that contained them.

In [8]:
numeric_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
 
to_drop = []
 
# drop label and attack
if "Label" in df.columns:
    to_drop.append("Label")
if "Attack" in df.columns:
    to_drop.append("Attack")
 
# drop 0/1 flag columns
flag_cols = [
    "FIN_Flag_Count","SYN_Flag_Count","RST_Flag_Count","PSH_Flag_Count",
    "ACK_Flag_Count","URG_Flag_Count","ECE_Flag_Count","Fwd_PSH_Flags"
]
 
for col in flag_cols:
    if col in numeric_cols:
        to_drop.append(col)
 
# drop constant columns 
for col in numeric_cols:
    if df[col].std() == 0:
        to_drop.append(col)
 
# drop columns with negative values
for col in numeric_cols:
    if df[col].min() < 0:
        to_drop.append(col)
 
# remove duplicates
to_drop = list(set(to_drop))
 
# final selected columns
selected_features = [c for c in numeric_cols if c not in to_drop]
 
print("Dropped columns:")
print(to_drop)
 
print("\nSelected features:")
print(selected_features)

Dropped columns:
['Attack', 'FIN_Flag_Count', 'Init_Win_bytes_forward', 'Label', 'PSH_Flag_Count', 'Init_Win_bytes_backward', 'Fwd_IAT_Min', 'URG_Flag_Count', 'Flow_IAT_Min', 'RST_Flag_Count', 'Fwd_PSH_Flags', 'ACK_Flag_Count', 'ECE_Flag_Count', 'SYN_Flag_Count']

Selected features:
['Destination_Port', 'Flow_Duration', 'Total_Fwd_Packets', 'Total_Backward_Packets', 'Total_Length_of_Fwd_Packets', 'Total_Length_of_Bwd_Packets', 'Fwd_Packet_Length_Max', 'Fwd_Packet_Length_Min', 'Fwd_Packet_Length_Mean', 'Fwd_Packet_Length_Std', 'Bwd_Packet_Length_Max', 'Bwd_Packet_Length_Min', 'Bwd_Packet_Length_Mean', 'Bwd_Packet_Length_Std', 'Flow_Bytes/s', 'Flow_Packets/s', 'Flow_IAT_Mean', 'Flow_IAT_Std', 'Flow_IAT_Max', 'Fwd_IAT_Total', 'Fwd_IAT_Mean', 'Fwd_IAT_Std', 'Fwd_IAT_Max', 'Bwd_IAT_Total', 'Bwd_IAT_Mean', 'Bwd_IAT_Std', 'Bwd_IAT_Max', 'Bwd_IAT_Min', 'Fwd_Header_Length', 'Bwd_Header_Length', 'Fwd_Packets/s', 'Bwd_Packets/s', 'Min_Packet_Length', 'Max_Packet_Length', 'Packet_Length_Mean', 'Pa

In [ ]:


# Create output directory
output_dir = Path("../eda/figures/univariate")
output_dir.mkdir(parents=True, exist_ok=True)

# Select a few features to analyze (start small!)
features_to_analyze = ['Destination_Port', 'Flow_Duration', 'Total_Fwd_Packets',
                        'Total_Backward_Packets', 'Total_Length_of_Fwd_Packets', 'Total_Length_of_Bwd_Packets', 
                        'Fwd_Packet_Length_Max', 'Fwd_Packet_Length_Min', 'Fwd_Packet_Length_Mean', 
                        'Fwd_Packet_Length_Std', 'Bwd_Packet_Length_Max', 'Bwd_Packet_Length_Min', 
                        'Bwd_Packet_Length_Mean', 'Bwd_Packet_Length_Std', 'Flow_Bytes/s', 'Flow_Packets/s', 
                        'Flow_IAT_Mean', 'Flow_IAT_Std', 'Flow_IAT_Max', 'Fwd_IAT_Total', 'Fwd_IAT_Mean', 
                        'Fwd_IAT_Std', 'Fwd_IAT_Max', 'Bwd_IAT_Total', 'Bwd_IAT_Mean', 'Bwd_IAT_Std', 
                        'Bwd_IAT_Max', 'Bwd_IAT_Min', 'Fwd_Header_Length', 'Bwd_Header_Length', 
                        'Fwd_Packets/s', 'Bwd_Packets/s', 'Min_Packet_Length', 'Max_Packet_Length', 
                        'Packet_Length_Mean', 'Packet_Length_Std', 'Packet_Length_Variance', 'Down/Up_Ratio', 
                        'Average_Packet_Size', 'Avg_Fwd_Segment_Size', 'Avg_Bwd_Segment_Size', 'Subflow_Fwd_Packets', 
                        'Subflow_Fwd_Bytes', 'Subflow_Bwd_Packets', 'Subflow_Bwd_Bytes', 'act_data_pkt_fwd', 
                        'min_seg_size_forward', 'Active_Mean', 'Active_Std', 'Active_Max', 'Active_Min', 'Idle_Mean', 
                        'Idle_Std', 'Idle_Max', 'Idle_Min']

print("Generating plots for selected features...\n")

# Generate plots for each feature
for feature in features_to_analyze:
    print(f"{feature}")
    
    # HISTOGRAM
    plt.figure(figsize=(10, 6))
    plt.hist(df[feature], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    plt.xlabel(feature)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {feature}')
    
    # Save histogram
    hist_filename = f"{feature.replace('/', '_')}_histogram.png"
    plt.savefig(output_dir / hist_filename, dpi=300, bbox_inches='tight')
    plt.close()
    
    # BOXPLOT
    plt.figure(figsize=(10, 6))
    plt.boxplot(df[feature], vert=True)
    plt.ylabel(feature)
    plt.title(f'Boxplot of {feature}')
    
    # Save boxplot
    box_filename = f"{feature.replace('/', '_')}_boxplot.png"
    plt.savefig(output_dir / box_filename, dpi=300, bbox_inches='tight')
    plt.close()
    
    # DETECT OUTLIERS (IQR Method)
    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[feature] < lower_bound) | (df[feature] > upper_bound)]
    outlier_pct = (len(outliers) / len(df)) * 100
    
    print(f"  Outliers: {len(outliers)} ({outlier_pct:.2f}%)")
    print(f"  Range: [{lower_bound:.2f}, {upper_bound:.2f}]\n")



Generating plots for selected features...

Destination_Port
  Outliers: 4417 (7.24%)
  Range: [80.00, 80.00]

Flow_Duration
  Outliers: 0 (0.00%)
  Range: [-128218994.00, 214163694.00]

Total_Fwd_Packets
  Outliers: 502 (0.82%)
  Range: [-7.00, 17.00]

Total_Backward_Packets
  Outliers: 336 (0.55%)
  Range: [-9.00, 15.00]

Total_Length_of_Fwd_Packets
  Outliers: 3201 (5.25%)
  Range: [-597.00, 995.00]

Total_Length_of_Bwd_Packets
  Outliers: 100 (0.16%)
  Range: [-17392.50, 28987.50]

Fwd_Packet_Length_Max
  Outliers: 563 (0.92%)
  Range: [-561.00, 935.00]

Fwd_Packet_Length_Min
  Outliers: 4841 (7.94%)
  Range: [0.00, 0.00]

Fwd_Packet_Length_Mean
  Outliers: 2768 (4.54%)
  Range: [-102.86, 171.43]

Fwd_Packet_Length_Std
  Outliers: 293 (0.48%)
  Range: [-235.28, 392.14]

Bwd_Packet_Length_Max
  Outliers: 0 (0.00%)
  Range: [-8688.00, 14480.00]

Bwd_Packet_Length_Min
  Outliers: 3575 (5.86%)
  Range: [0.00, 0.00]

Bwd_Packet_Length_Mean
  Outliers: 0 (0.00%)
  Range: [-2898.75, 4831.2

12/2/2025 - Moosa - Univariate Data Analysis Report

**Purpose:**  
The goal is to see how the data is shaped, find outliers, and clean anything that doesn’t make sense before we move forward.

**Interpretation / Findings:** 
- cleaned the dataset by removing columns that were constant, contained only 0/1 values, or had negative values. After selecting the useful numeric features, we generated histograms and boxplots for each one and saved all the plots in the eda/figures/univariate/ folder.
- Also calculated outliers using the IQR method to see which features had extreme values. This helped us understand the spread of each feature and identify any unusual patterns in the data.

### Done with Task

In [10]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("../data/cleaned/wednesday_cleaned.csv")

# Get numeric columns only
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [col for col in numeric_cols if col not in ['Attack', 'Label']]

# Store outlier statistics
outlier_data = []

# Calculate outliers for each feature
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    
    outlier_data.append({
        'Feature': col,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'Lower_Bound': lower_bound,
        'Upper_Bound': upper_bound,
        'Num_Outliers': len(outliers),
        'Outlier_Percentage': (len(outliers) / len(df)) * 100
    })

# Create DataFrame
outlier_df = pd.DataFrame(outlier_data)

# Sort by outlier percentage (highest first)
outlier_df = outlier_df.sort_values('Outlier_Percentage', ascending=False)

# Display the table
print(outlier_df.to_string(index=False))



                    Feature            Q1           Q3          IQR   Lower_Bound  Upper_Bound  Num_Outliers  Outlier_Percentage
       min_seg_size_forward     32.000000 3.200000e+01 0.000000e+00  3.200000e+01 3.200000e+01         16751           27.460206
             Flow_Packets/s      0.131970 3.178741e+01 3.165544e+01 -4.735118e+01 7.927056e+01         13670           22.409469
              Fwd_Packets/s      0.079042 1.492938e+01 1.485034e+01 -2.219647e+01 3.720490e+01         13594           22.284881
                 Active_Max      0.000000 1.997000e+03 1.997000e+03 -2.995500e+03 4.992500e+03         13417           21.994721
                Active_Mean      0.000000 1.997000e+03 1.997000e+03 -2.995500e+03 4.992500e+03         13417           21.994721
                 Active_Min      0.000000 1.990000e+03 1.990000e+03 -2.985000e+03 4.975000e+03         13122           21.511123
               Flow_IAT_Min      1.000000 5.100000e+01 5.000000e+01 -7.400000e+01 1.260000e+02   

12/2/2025 - Nafisa - Outlier Table Report

**Purpose:**  
I generate summary table of outliers for each numeric feature using the IQR method.

**Interpretation / Findings:** 
- The data is presented in a table format showing the number of outliers for each feature in decresing order of outlier count.

### Done with Task